# 03 – Walk-forward backtest and metrics

Runs `src/backtest.py` for the four portfolios and checks the result.
Timing and cost conventions: `docs/methodology.md`, section 7.

In [1]:
# Setup
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
sys.path.insert(0, str(ROOT))
from src import backtest, data
from src.optimizer import BENCH, check_rules

TICKERS = data.TICKERS
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

## 1. Short sample first (2016)

In [2]:
daily, monthly = backtest.run_backtest("full", first="2015-12", last="2016-11")
print(f"Sessions: {daily.index[0].date()} -> {daily.index[-1].date()} | {len(daily)}")
print(monthly[["traded", "cost", "te_ex_ante", "relaxed_turnover_cap"]].round(4))

Sessions: 2016-01-04 -> 2016-12-30 | 252
         traded    cost  te_ex_ante relaxed_turnover_cap
month                                                   
2015-12  0.2072  0.0001      0.0170                 None
2016-01  0.2004  0.0001      0.0200                 None
2016-02  0.0992  0.0000      0.0200                 None
2016-03  0.1151  0.0001      0.0200                 None
2016-04  0.1761  0.0001      0.0200                 None
2016-05  0.0123  0.0000      0.0200                 None
2016-06  0.0347  0.0000      0.0200                 None
2016-07  0.1558  0.0001      0.0200                 None
2016-08  0.1405  0.0001      0.0200                 None
2016-09  0.0123  0.0000      0.0200                 None
2016-10  0.2049  0.0001      0.0200                 None
2016-11  0.2012  0.0001      0.0143                 None


## 2. Full run: benchmark, BL with TE budget only, BL under full mandate, naive MVO (saved to `data/results/`)

In [3]:
net, gross, log = backtest.run_all()
print(f"Evaluation: {net.index[0].date()} -> {net.index[-1].date()} | {len(net)} sessions")
print("Files:", sorted(p.name for p in backtest.RESULTS_DIR.glob("*.csv")))

Evaluation: 2016-01-04 -> 2026-10-30 | 2723 sessions
Files: ['daily_returns_gross.csv', 'daily_returns_net.csv', 'metrics.csv', 'metrics_by_year.csv', 'monthly_log.csv', 'te_by_block.csv', 'variants.csv']


## 3. Checks

In [4]:
W = log.set_index(["portfolio", "month"])[[f"w_{t}" for t in TICKERS]].set_axis(TICKERS, axis=1)
rules = {"benchmark": "full", "bl_te_only": "long_only", "bl_full": "full", "mvo_naive": "full"}
viol = sum(bool(check_rules(x, rules[p])) for (p, _), x in W.iterrows())
print(f"Weights sum to 1: max error {(W.sum(axis=1) - 1).abs().max():.1e} | rule violations: {viol}")

relaxed = log.dropna(subset=["relaxed_turnover_cap"])
print("Turnover cap relaxed:", relaxed[["portfolio", "month", "relaxed_turnover_cap"]].to_dict("records"))

cost_gap = ((1 + gross).prod() / (1 + net).prod() - 1)
print("Cumulative cost drag:", cost_gap.round(5).to_dict())

# Benchmark vs static weights on monthly returns: gap = intra-month drift + next-day rebalance only
_, _, mon = data.load_processed()
static = mon.loc["2016-01":"2026-10", TICKERS] @ BENCH
realized = (1 + net["benchmark"]).groupby(net.index.to_period("M")).prod() - 1
print(f"Benchmark vs static monthly: mean |gap| {(realized - static).abs().mean():.2e}, "
      f"max {(realized - static).abs().max():.2e} ({(realized - static).abs().idxmax()})")

Weights sum to 1: max error 1.1e-15 | rule violations: 0
Turnover cap relaxed: [{'portfolio': 'bl_te_only', 'month': Period('2020-03', 'M'), 'relaxed_turnover_cap': 0.3125}]
Cumulative cost drag: {'benchmark': 0.0, 'bl_te_only': 0.0057, 'bl_full': 0.00535, 'mvo_naive': 0.00857}
Benchmark vs static monthly: mean |gap| 1.52e-04, max 4.80e-03 (2020-04)


## 4. Metrics

`src/metrics.py`, saved to `data/results/metrics.csv` and `metrics_by_year.csv` for the app.
Net of 5 bp costs; annualized with 252 sessions; ES as a positive daily loss.

In [5]:
from src import metrics

table, yearly = metrics.compute_all()
print(table.T.round(4))

                                  benchmark  bl_te_only  bl_full  mvo_naive
Return (CAGR)                        0.0817      0.0901   0.0864     0.0907
Volatility                           0.1042      0.1090   0.1097     0.1073
Sharpe                               0.5855      0.6351   0.6008     0.6487
Active return (net)                  0.0000      0.0082   0.0049     0.0086
Active return (gross)                0.0000      0.0088   0.0054     0.0094
Active CAGR                          0.0000      0.0084   0.0047     0.0090
Realized TE                          0.0000      0.0229   0.0218     0.0235
Realized TE (monthly)                0.0000      0.0198   0.0184     0.0202
Information ratio                       NaN      0.3599   0.2246     0.3646
t-stat active return                    NaN      1.1831   0.7383     1.1987
Jensen alpha (annualized)               NaN      0.0084   0.0044     0.0085
Jensen beta                          1.0000      0.9920   1.0018     1.0002
t-stat Jense

Realized TE and active return by calendar year (is the 2% budget respected out of sample?)

In [6]:
print(yearly.round(4))

     Active return                   Realized TE                  
        bl_te_only bl_full mvo_naive  bl_te_only bl_full mvo_naive
Date                                                              
2016       -0.0192 -0.0137   -0.0357      0.0196  0.0197    0.0212
2017        0.0362  0.0307    0.0247      0.0170  0.0171    0.0165
2018       -0.0170 -0.0230   -0.0178      0.0288  0.0283    0.0276
2019       -0.0073 -0.0075   -0.0060      0.0175  0.0179    0.0179
2020        0.0331 -0.0031    0.0382      0.0326  0.0228    0.0349
2021        0.0091  0.0113    0.0101      0.0179  0.0177    0.0178
2022       -0.0035 -0.0053    0.0014      0.0233  0.0230    0.0235
2023        0.0048  0.0095    0.0062      0.0186  0.0186    0.0187
2024        0.0385  0.0231    0.0353      0.0214  0.0208    0.0202
2025        0.0130  0.0293    0.0293      0.0207  0.0216    0.0246
2026        0.0018  0.0023    0.0087      0.0294  0.0298    0.0299
